# Notebook 02b — ID Market Pre-Gate

**Across the Water: French Nuclear Unplanned Outages and GB Power Prices**

**This is the project's first kill switch.**

Tests whether unplanned French nuclear outage announcements produce a
detectable price reaction in the GB intraday (ID) market in the hours
following announcement.

---

### Pre-registered gate condition

On days with an unplanned outage announced between 14:00 and 16:00 D-2,
test whether GB ID hourly settlement prices move significantly in the
**16:00–17:00 D-2 window** relative to control days.

| | |
|---|---|
| **Signal** | Unplanned FR nuclear outage announced ~15:00 D-2 |
| **Primary window** | 16:00–17:00 D-2 (first entirely post-announcement bar) |
| **Extended window** | 15:00–20:00 D-2 cumulative (reported, not gated) |
| **H₀** | β = 0 (no significant ID reaction) |
| **H₁** | β > 0, one-tailed |
| **Gate threshold** | β > 0, p < 0.10 for 16:00–17:00 window |
| **If gate fails** | Stop. Report upper bound on undetected effect. DA gate not run. |
| **No exceptions** | |

---

### Data constraint (pre-registered)

This test uses EPEX GB **hourly** ID settlement prices — free public data.
No tick data, no order book depth, no sub-hourly fills.
Results stated as point estimates with CI at 1 MW. Not extrapolated.


> 📋 **NOT RUN** — Pre-registered but not executed.
> The DA hard gate (Notebook 02) failed before this gate was reached.
> Published unmodified to demonstrate pre-registration discipline.

**Data availability note (added post-pipeline build):** 

EPEX GB hourly ID settlement prices are listed as free and public on the EPEX website but in practice require either manual download of individual monthly report files or a paid API subscription. Given the marginal value of the hourly ID test (low resolution, high bid-offer on outage days, likely null result as noted in the prior), Notebook 02b is skipped pending data availability. The DA hard gate (Notebook 02) runs directly. If EPEX ID data becomes available, 02b can be inserted before 02 without affecting downstream notebooks.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats

from src import log, load, DATA_DIR, MAIN_START, MAIN_END
from src.utils import add_season_col

pd.set_option("display.float_format", "{:.4f}".format)
print("Setup complete.")


---
## 1. Load Data

Load ENTSO-E unavailability notices and EPEX GB hourly ID prices.


In [ ]:
# ── Load ENTSO-E unavailability ────────────────────────────────────────────
entso = load("entso_unavailability")

# Filter to unplanned outages only
unplanned = entso[entso["outage_type"] == "unplanned"].copy()

print(f"Total unplanned outage records: {len(unplanned):,}")
print(f"Date range: {unplanned.index.min().date()} → {unplanned.index.max().date()}")
print(unplanned.head(3))


In [ ]:
# ── Load EPEX GB hourly ID prices ─────────────────────────────────────────
try:
    epex_id = load("epex_gb_id_hourly")
    print(f"EPEX GB ID hourly: {len(epex_id):,} rows")
    print(f"Date range: {epex_id.index.min().date()} → {epex_id.index.max().date()}")
    EPEX_AVAILABLE = True
except FileNotFoundError:
    print("⚠️  epex_gb_id_hourly.parquet not found.")
    print("   Download EPEX GB hourly settlement price CSVs from:")
    print("   https://www.epexspot.com/en/market-data#trading-results-downloads")
    print("   Place in data/epex_gb_id/ and re-run Notebook 01 Section 7.")
    print()
    print("   Running in SIMULATION MODE — using GB DA price as a proxy.")
    print("   Real results require actual EPEX ID data.")
    EPEX_AVAILABLE = False
    
    # Use GB DA prices as a proxy to demonstrate the methodology
    da_prices = load("elexon_da_prices")
    epex_id = da_prices[["gb_da_price"]].rename(columns={"gb_da_price": "gb_id_price_gbp_mwh"})
    epex_id = epex_id.resample("h").ffill()
    print(f"   Proxy ID prices: {len(epex_id):,} rows")


---
## 2. Build Event Dataset

### Event identification logic

An "event day" is a calendar day on which at least one unplanned outage notice
for a French nuclear unit was first published between **14:00 and 16:00 UTC D-2**.

Using the `fetch_timestamp` rather than the event start time — backtest integrity.

Window construction:
- The 15:00–16:00 bar contains pre-announcement trading on most days (announcement
  often at ~15:05). The **16:00–17:00 bar** is the first entirely post-announcement
  settlement period.
- The 15:00–16:00 bar is retained as a robustness check only.


In [ ]:
def build_event_days(unplanned_df: pd.DataFrame) -> pd.DataFrame:
    """
    Identify event days: dates where an unplanned outage was first announced
    between 14:00 and 16:00 UTC.
    
    Uses fetch_timestamp for backtest integrity.
    Returns DataFrame indexed by announcement date with outage_mw column.
    """
    df = unplanned_df.copy()
    
    # Parse fetch_timestamp to datetime
    df["fetch_dt"] = pd.to_datetime(df["fetch_timestamp"], utc=True, errors="coerce")
    
    # Announcement window: 14:00–16:00 UTC
    df["fetch_hour"] = df["fetch_dt"].dt.hour
    in_window = df[(df["fetch_hour"] >= 14) & (df["fetch_hour"] < 16)].copy()
    
    # Aggregate to daily: sum unavailable MW per announcement date
    in_window["announcement_date"] = in_window["fetch_dt"].dt.normalize()
    
    events = (
        in_window
        .groupby("announcement_date")["unavailable_mw"]
        .sum()
        .rename("outage_mw_announced")
        .reset_index()
    )
    events["announcement_date"] = pd.to_datetime(events["announcement_date"], utc=True)
    events = events.set_index("announcement_date")
    
    return events

events = build_event_days(unplanned)
print(f"Event days (unplanned outage announced 14:00–16:00 UTC): {len(events)}")
print(f"Date range: {events.index.min().date()} → {events.index.max().date()}")
print(f"Mean outage MW announced on event days: {events['outage_mw_announced'].mean():.0f} MW")
print()
print(events.describe())


In [ ]:
# ── Build ID price change features ───────────────────────────────────────────
def extract_hourly_price_windows(epex_id_df: pd.DataFrame) -> pd.DataFrame:
    """
    For each date, extract:
      - p_1516: settlement price 15:00–16:00 (baseline / robustness)
      - p_1617: settlement price 16:00–17:00 (primary test window)
      - p_1720: mean settlement price 17:00–20:00 (extended window)
      - delta_1617: p_1617 - p_1516 (primary test metric)
      - delta_1520: p_1720 - p_1516 (extended)
    """
    df = epex_id_df[["gb_id_price_gbp_mwh"]].copy()
    
    records = []
    for date, group in df.groupby(df.index.normalize()):
        def price_at(h_start):
            mask = (group.index.hour == h_start)
            vals = group.loc[mask, "gb_id_price_gbp_mwh"].dropna()
            return vals.mean() if len(vals) > 0 else float("nan")
        
        p_1516 = price_at(15)
        p_1617 = price_at(16)
        p_1720 = group.loc[
            (group.index.hour >= 17) & (group.index.hour < 20),
            "gb_id_price_gbp_mwh"
        ].mean()
        
        records.append({
            "date":        pd.Timestamp(date, tz="UTC"),
            "p_1516":      p_1516,
            "p_1617":      p_1617,
            "p_1720":      p_1720,
            "delta_1617":  p_1617 - p_1516,   # primary metric
            "delta_1520":  p_1720 - p_1516,    # extended metric
        })
    
    return pd.DataFrame(records).set_index("date").sort_index()

price_windows = extract_hourly_price_windows(epex_id)
print(f"Price windows built for {len(price_windows)} dates")
price_windows.describe()


In [ ]:
# ── Merge events with price windows ──────────────────────────────────────────
# Events are announced at D-2: the outage announced on date T affects
# price windows also on date T (within the same day).
# The DA gate uses D-2 → next DA auction structure; here we test same-day ID reaction.

panel = price_windows.copy()
panel = add_season_col(panel)

# Merge event flags
panel = panel.join(events[["outage_mw_announced"]], how="left")
panel["is_event"] = panel["outage_mw_announced"].notna().astype(int)
panel["outage_mw"] = panel["outage_mw_announced"].fillna(0)

# Restrict to main sample period (post IFA2 coupling)
panel = panel[panel.index >= pd.Timestamp(MAIN_START, tz="UTC")]

# Drop rows with missing primary outcome
panel = panel.dropna(subset=["delta_1617"])

n_events   = panel["is_event"].sum()
n_control  = (panel["is_event"] == 0).sum()

print(f"Panel: {len(panel)} trading days ({MAIN_START} → {panel.index.max().date()})")
print(f"  Event days (outage 14:00–16:00):  {n_events}")
print(f"  Control days:                      {n_control}")
print(f"  Event rate: {n_events/len(panel)*100:.1f}%")


---
## 3. Pre-Gate Regression

**Specification (Model A — fr_da_price not yet available):**

```
delta_id_price(16:00–17:00, t) = α
                                + β × outage_mw_announced(t)
                                + season_fe
                                + u(t)
```

HAC standard errors, 12 lags (hourly data, daily observations).

Gate threshold: **β > 0, p < 0.10** (one-tailed).


In [ ]:
def run_id_gate_regression(
    panel: pd.DataFrame,
    outcome: str = "delta_1617",
    label: str = "16:00–17:00 primary window",
) -> dict:
    """Run the pre-gate OLS regression and return results dict."""
    df = panel.dropna(subset=[outcome, "outage_mw"]).copy()
    
    # Season dummies
    season_dummies = pd.get_dummies(df["season"], prefix="season", drop_first=True)
    
    X = pd.concat([
        df[["outage_mw"]],
        season_dummies,
    ], axis=1).astype(float)
    X = sm.add_constant(X)
    y = df[outcome].astype(float)
    
    model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 12})
    
    beta    = model.params["outage_mw"]
    se      = model.bse["outage_mw"]
    t_stat  = model.tvalues["outage_mw"]
    p_two   = model.pvalues["outage_mw"]
    p_one   = p_two / 2 if beta > 0 else 1.0 - p_two / 2
    ci_lo   = beta - 1.96 * se
    ci_hi   = beta + 1.96 * se
    
    return {
        "window":    label,
        "outcome":   outcome,
        "N":         int(model.nobs),
        "beta":      beta,
        "se":        se,
        "t_stat":    t_stat,
        "p_two":     p_two,
        "p_one":     p_one,
        "ci_lo":     ci_lo,
        "ci_hi":     ci_hi,
        "r2":        model.rsquared,
        "model":     model,
    }

# Primary window: 16:00–17:00
res_primary = run_id_gate_regression(panel, "delta_1617", "16:00–17:00 (primary)")

# Extended window: 15:00–20:00
res_extended = run_id_gate_regression(panel, "delta_1520", "15:00–20:00 (extended)")

# Robustness: 15:00–16:00 (partially contaminated)
res_robustness = run_id_gate_regression(panel, "delta_1617", "15:00–16:00 (robustness — partially contaminated)")

results = [res_primary, res_extended, res_robustness]


In [ ]:
# ── Gate decision ─────────────────────────────────────────────────────────────
r = res_primary

print("=" * 65)
print("ID MARKET PRE-GATE — NOTEBOOK 02b")
print("=" * 65)
print()
print(f"Window:         {r['window']}")
print(f"N observations: {r['N']}")
print(f"β (outage MW):  {r['beta']:.5f}  £/MWh per MW of outage")
print(f"                = {r['beta']*1000:.3f}  £/MWh per GW of outage")
print(f"SE:             {r['se']:.5f}")
print(f"t-statistic:    {r['t_stat']:.3f}")
print(f"p-value (two):  {r['p_two']:.4f}")
print(f"p-value (one):  {r['p_one']:.4f}  ← gate uses this")
print(f"95% CI:         [{r['ci_lo']:.5f}, {r['ci_hi']:.5f}]")
print(f"R²:             {r['r2']:.4f}")
print()

# Gate evaluation
gate_beta_pass = r["beta"] > 0
gate_p_pass    = r["p_one"] < 0.10
gate_pass      = gate_beta_pass and gate_p_pass

print("─" * 65)
print("GATE CONDITIONS:")
print(f"  β > 0:        {'✅ PASS' if gate_beta_pass else '❌ FAIL'}  (β = {r['beta']:.5f})")
print(f"  p < 0.10 (1-tail): {'✅ PASS' if gate_p_pass else '❌ FAIL'}  (p = {r['p_one']:.4f})")
print()

if gate_pass:
    print("✅  ID PRE-GATE: PASSED")
    print()
    print("Point estimate: £{:.3f}/MWh per GW of unplanned outage (95% CI: [{:.3f}, {:.3f}])".format(
        r["beta"]*1000, r["ci_lo"]*1000, r["ci_hi"]*1000
    ))
    print()
    print("Note: hourly fill assumptions are imperfect. Bid-offer spreads on")
    print("outage days (~£0.50–0.80/MWh round trip) may consume this signal.")
    print()
    print("→ Proceed to Notebook 02 (DA hard gate).")
else:
    print("❌  ID PRE-GATE: FAILED")
    print()
    print("The market produces no detectable hourly ID reaction at p < 0.10.")
    print("This does not prove no signal exists — ID markets react in minutes,")
    print("not hours, and hourly settlement prices can wash out sub-hourly moves.")
    print()
    print("Upper bound on undetected hourly effect:")
    print(f"  β ≤ {r['ci_hi']:.5f} £/MWh per MW  (95% CI upper bound)")
    print(f"    = {r['ci_hi']*1000:.3f} £/MWh per GW")
    print()
    print("→ Project stops here. DA gate will not be run.")
    print("  See 'If the Gate Fails' section in README for interpretation.")
    
print("─" * 65)
GATE_PASSED = gate_pass


In [ ]:
# ── Extended results table ────────────────────────────────────────────────────
print("\nAll windows:")
print(f"{'Window':<35} {'β/MW':>10} {'β/GW':>10} {'p (1-tail)':>12} {'Gate':>8}")
print("─" * 78)
for r in results:
    gate_flag = "← GATE" if r["window"] == res_primary["window"] else ""
    print(f"{r['window']:<35} {r['beta']:>10.5f} {r['beta']*1000:>10.3f} {r['p_one']:>12.4f} {gate_flag}")


---
## 4. Diagnostic Plots


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Distribution of delta_1617 — event vs control
ax = axes[0, 0]
event_deltas   = panel.loc[panel["is_event"]==1, "delta_1617"].dropna()
control_deltas = panel.loc[panel["is_event"]==0, "delta_1617"].dropna()
ax.hist(control_deltas, bins=40, alpha=0.6, color="steelblue", label=f"Control (n={len(control_deltas)})")
ax.hist(event_deltas,   bins=20, alpha=0.7, color="tomato",    label=f"Event (n={len(event_deltas)})")
ax.axvline(0, color="black", lw=0.8, ls="--")
ax.set_title("Δ ID Price 16:00–17:00: Event vs Control Days")
ax.set_xlabel("£/MWh price change")
ax.legend()

# 2. Scatter: outage MW vs delta_1617
ax = axes[0, 1]
event_panel = panel[panel["is_event"] == 1].dropna(subset=["outage_mw", "delta_1617"])
ax.scatter(event_panel["outage_mw"]/1000, event_panel["delta_1617"],
           alpha=0.5, color="tomato", s=30)
if len(event_panel) > 2:
    z = np.polyfit(event_panel["outage_mw"]/1000, event_panel["delta_1617"], 1)
    xr = np.linspace(event_panel["outage_mw"].min()/1000, event_panel["outage_mw"].max()/1000, 100)
    ax.plot(xr, np.polyval(z, xr), color="darkred", lw=1.5, label="OLS fit")
ax.axhline(0, color="black", lw=0.8, ls="--")
ax.set_xlabel("Outage MW announced (GW)")
ax.set_ylabel("Δ ID price 16:00–17:00 (£/MWh)")
ax.set_title("Outage Size vs ID Price Reaction")
ax.legend()

# 3. Rolling 90-day mean delta_1617 — event vs control
ax = axes[1, 0]
for label, subset, color in [
    ("Event", panel[panel["is_event"]==1], "tomato"),
    ("Control", panel[panel["is_event"]==0], "steelblue"),
]:
    rolling = subset["delta_1617"].rolling(90, min_periods=10).mean()
    ax.plot(rolling.index, rolling, lw=1.2, color=color, label=label)
ax.axhline(0, color="black", lw=0.8, ls="--")
ax.set_title("Rolling 90-day Mean Δ ID Price (16:00–17:00)")
ax.set_ylabel("£/MWh")
ax.legend()

# 4. Average price by hour — event vs control
ax = axes[1, 1]
epex_w_events = epex_id.copy()
epex_w_events["hour"] = epex_w_events.index.hour
epex_w_events["date"] = epex_w_events.index.normalize()
epex_w_events = epex_w_events.merge(
    panel[["is_event"]].reset_index().rename(columns={"index": "date", "date": "date"}),
    left_on="date", right_on="date", how="left"
)
for label, flag, color in [("Event", 1, "tomato"), ("Control", 0, "steelblue")]:
    subset = epex_w_events[epex_w_events["is_event"]==flag]
    hourly_mean = subset.groupby("hour")["gb_id_price_gbp_mwh"].mean()
    ax.plot(hourly_mean.index, hourly_mean.values, lw=1.5, color=color, label=label, marker="o", ms=4)
ax.axvspan(15, 17, alpha=0.1, color="orange", label="Announcement window")
ax.set_title("Mean Hourly ID Price: Event vs Control")
ax.set_xlabel("Hour (UTC)")
ax.set_ylabel("£/MWh")
ax.legend()

plt.suptitle("Notebook 02b — ID Market Pre-Gate Diagnostics", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(DATA_DIR / "plot_02b_id_gate.png", dpi=120, bbox_inches="tight")
plt.show()
print("Plots saved → data/plot_02b_id_gate.png")


---
## 5. Minimum Detectable Effect (reported regardless of gate outcome)

Per the pre-registered design, the MDE and implied outage size threshold
for tradeability are reported whether or not the gate passes.


In [ ]:
r = res_primary

# MDE at 80% power, α=0.10 one-tailed
z_alpha = 1.282  # one-tailed 10%
z_power = 0.842  # 80% power
sigma   = r["se"] * np.sqrt(r["N"])
mde_mw  = (z_alpha + z_power) * sigma / np.sqrt(r["N"])

print("MINIMUM DETECTABLE EFFECT (80% power, α=0.10 one-tailed):")
print(f"  MDE β = {mde_mw:.5f} £/MWh per MW")
print(f"        = {mde_mw*1000:.3f} £/MWh per GW")
print()

# Bid-offer spread on outage days: ~£0.50–0.80/MWh round trip
bo_lo, bo_hi = 0.50, 0.80
print("Tradeability threshold (bid-offer spread on outage days: £0.50–0.80/MWh):")
min_outage_lo = bo_lo / (mde_mw * 1000) if mde_mw > 0 else float("inf")
min_outage_hi = bo_hi / (mde_mw * 1000) if mde_mw > 0 else float("inf")
print(f"  Minimum outage size to clear spread (optimistic):    {min_outage_lo:.0f} MW")
print(f"  Minimum outage size to clear spread (conservative):  {min_outage_hi:.0f} MW")
print()
print("Note: typical single reactor trip = 900–1,400 MW.")
print("If MDE > typical outage size, the strategy is unlikely to clear costs")
print("at hourly resolution with public data alone.")
print()
print(f"Actual estimated β: {r['beta']*1000:.3f} £/MWh per GW  (p={r['p_one']:.4f}, one-tailed)")


---
## Summary

| Metric | Value |
|---|---|
| **Gate result** | See cell output above |
| Primary window (16:00–17:00) | β, CI, p reported above |
| Extended window (15:00–20:00) | Reported, not gated |
| MDE (80% power) | Reported above |
| Data constraint | Hourly settlement prices only — no tick data |
| Bid-offer on outage days | ~£0.50–0.80/MWh round trip |

**If gate passed:** proceed to `02_da_gate.ipynb`.  
**If gate failed:** project stops here. See README § "If the Gate Fails".
